# Advanced Thermodynamics: Generating Vapor-Liquid Phase Diagrams

**Objective:** This lesson moves beyond using existing VLE data to generating it from first principles. We will construct a complete, publication-quality Temperature-Composition (T-x-y) diagram for a binary mixture using fundamental thermodynamic models (Raoult's Law and the Antoine Equation).

**Learning Goals:**
1.  Review the principles of Vapor-Liquid Equilibrium (VLE) for ideal binary systems.
2.  Implement the **Antoine Equation** to calculate temperature-dependent vapor pressures.
3.  Develop algorithms for **Bubble Point** and **Dew Point** calculations, which require numerical root-finding.
4.  Combine these algorithms to generate the data for the 'bubble line' and 'dew line'.
5.  Construct and interpret a complete T-x-y diagram, visualizing the liquid, vapor, and two-phase regions.

## Part 1: The Thermodynamic Model

For an ideal binary mixture of components 1 and 2, the VLE is governed by **Raoult's Law** for each component:
$$ y_1 P = x_1 P_1^{sat}(T) \quad , \quad y_2 P = x_2 P_2^{sat}(T) $$
Since $y_1 + y_2 = 1$, we can sum these equations to get an expression for the total pressure, $P$:
$$ P = x_1 P_1^{sat}(T) + x_2 P_2^{sat}(T) $$

The temperature dependence of the vapor pressures, $P_i^{sat}$, is described by the **Antoine Equation**:
$$ \log_{10}(P_i^{sat}) = A_i - \frac{B_i}{C_i + T} $$

## Part 2: Bubble and Dew Point Calculations

A T-x-y diagram is plotted at a constant pressure. Our task is to find the temperature at which a phase transition occurs for a given composition.

**1. Bubble Point Temperature ($T_{bub}$):**
*   **Problem:** Given a liquid composition ($x_1, x_2$) and a total pressure ($P$), find the temperature ($T$) at which the first bubble of vapor forms, and find the composition of that vapor ($y_1, y_2$).
*   **Method:** We need to find the temperature $T$ that solves the equation:
$$ P = x_1 P_1^{sat}(T) + x_2 P_2^{sat}(T) \implies x_1 P_1^{sat}(T) + x_2 P_2^{sat}(T) - P = 0 $$
This is a non-linear algebraic equation that we must solve numerically for $T$.

**2. Dew Point Temperature ($T_{dew}$):**
*   **Problem:** Given a vapor composition ($y_1, y_2$) and a total pressure ($P$), find the temperature ($T$) at which the first drop of liquid forms, and find the composition of that liquid ($x_1, x_2$).
*   **Method:** From Raoult's Law, $x_i = y_i P / P_i^{sat}(T)$. Since $\sum x_i = 1$, we must find the temperature $T$ that solves:
$$ \frac{y_1 P}{P_1^{sat}(T)} + \frac{y_2 P}{P_2^{sat}(T)} = 1 \implies \frac{y_1 P}{P_1^{sat}(T)} + \frac{y_2 P}{P_2^{sat}(T)} - 1 = 0 $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

# --- Define System Parameters ---
# We will model the Benzene(1) - Toluene(2) system
P_total = 1.0 # System pressure in atm

# Antoine Coefficients for P_sat in mmHg and T in Celsius
# antoine_coeffs = [A, B, C]
antoine_benzene = [6.90565, 1211.033, 220.79]
antoine_toluene = [6.95464, 1344.8,   219.482]

def antoine_pressure(T_celsius, coeffs):
    """Calculates vapor pressure in atm given T in Celsius."""
    A, B, C = coeffs
    P_mmHg = 10**(A - (B / (C + T_celsius)))
    return P_mmHg / 760.0 # Convert to atm

print("System parameters and Antoine function are defined.")

In [ ]:
# --- Bubble and Dew Point Functions ---

def bubble_point_T(x1, P_total):
    """Calculates bubble point temperature and vapor composition y1."""
    x2 = 1 - x1
    
    # Define the equation to solve: f(T) = 0
    def equation_to_solve(T):
        P1_sat = antoine_pressure(T, antoine_benzene)
        P2_sat = antoine_pressure(T, antoine_toluene)
        return x1 * P1_sat + x2 * P2_sat - P_total
    
    # Good initial guess is the weighted average of pure component boiling points
    # T_boil_benzene at 1atm is ~80.1 C, T_boil_toluene is ~110.6 C
    initial_guess = x1 * 80.1 + x2 * 110.6
    
    T_bubble = fsolve(equation_to_solve, initial_guess)[0]
    
    # Now calculate vapor composition y1
    y1 = (x1 * antoine_pressure(T_bubble, antoine_benzene)) / P_total
    return T_bubble, y1

def dew_point_T(y1, P_total):
    """Calculates dew point temperature and liquid composition x1."""
    y2 = 1 - y1
    
    # Define the equation to solve: f(T) = 0
    def equation_to_solve(T):
        P1_sat = antoine_pressure(T, antoine_benzene)
        P2_sat = antoine_pressure(T, antoine_toluene)
        # Avoid division by zero if pressures are too low
        P1_sat = max(P1_sat, 1e-9)
        P2_sat = max(P2_sat, 1e-9)
        return (y1 / P1_sat) + (y2 / P2_sat) - (1 / P_total)
    
    initial_guess = y1 * 80.1 + y2 * 110.6
    T_dew = fsolve(equation_to_solve, initial_guess)[0]
    
    # Now calculate liquid composition x1
    x1 = (y1 * P_total) / antoine_pressure(T_dew, antoine_benzene)
    return T_dew, x1

print("Bubble and Dew point solver functions are defined.")

## Part 3: Generating and Plotting the T-x-y Diagram
Now we loop through all possible compositions to generate the data for our bubble and dew lines.

In [ ]:
# --- Generate Data ---
x_values = np.linspace(0, 1, 50)
y_values = np.linspace(0, 1, 50)

bubble_line_T = []
bubble_line_y = []
dew_line_T = []
dew_line_x = []

# Calculate Bubble Line (T vs x)
for x1 in x_values:
    T_b, y1 = bubble_point_T(x1, P_total)
    bubble_line_T.append(T_b)
    bubble_line_y.append(y1)

# Calculate Dew Line (T vs y)
for y1 in y_values:
    T_d, x1 = dew_point_T(y1, P_total)
    dew_line_T.append(T_d)
    dew_line_x.append(x1)

# --- Plotting the Diagram ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 8))

plt.plot(x_values, bubble_line_T, 'b-', linewidth=3, label='Bubble Line (Saturated Liquid)')
# For the dew line, we plot T vs x, so we use the calculated dew_line_x values
plt.plot(dew_line_x, dew_line_T, 'r-', linewidth=3, label='Dew Line (Saturated Vapor)')

# Add annotations
plt.title(f'T-x-y Diagram for Benzene-Toluene at {P_total} atm', fontsize=16, weight='bold')
plt.xlabel('Mole Fraction Benzene (x, y)', fontsize=14)
plt.ylabel('Temperature (°C)', fontsize=14)
plt.xlim(0, 1)
plt.legend()
plt.grid(True)

plt.text(0.2, 95, 'Subcooled Liquid', fontsize=12)
plt.text(0.4, 102, 'Two-Phase Region', fontsize=12, rotation=-20)
plt.text(0.7, 90, 'Superheated Vapor', fontsize=12)

plt.show()

### Interpreting the Diagram

This diagram is a complete map of the VLE behavior for our system at 1 atm.
*   **Bubble Line (Blue):** If you take a liquid of a certain composition (e.g., x=0.4) and start heating it, this is the temperature at which it will begin to boil.
*   **Dew Line (Red):** If you take a vapor of a certain composition (e.g., y=0.6) and start cooling it, this is the temperature at which it will begin to condense.
*   **Two-Phase Region:** Between the two lines, liquid and vapor coexist in equilibrium. A horizontal tie-line connects the liquid composition (on the bubble line) and the vapor composition (on the dew line) that are in equilibrium at that temperature.

## Student Challenges

1.  **Effect of Pressure:** How does pressure affect the phase diagram? Re-run the entire simulation, but change the system pressure `P_total` to `2.0` atm. Does the entire diagram shift up or down in temperature? Why?

2.  **Azeotropes (Advanced):** Some systems, like Ethanol-Water, are non-ideal and exhibit an **azeotrope** where the bubble and dew lines touch. The model we used (Raoult's Law) cannot predict this. Research a more advanced VLE model, like the **Modified Raoult's Law** with activity coefficients (e.g., from the Wilson equation). Can you implement this more complex model to generate a T-x-y diagram for a non-ideal system?